# Forecast Stability | NeuralForecast
> Measure how much a model revises its forecasts as new information arrives, and whether those revisions were worth making.

## Motivation

Forecasts are rarely produced once. In most production settings a model is re-run on a schedule, and each run issues a new forecast for a horizon that overlaps the previous one. A date three weeks out gets predicted this week, again next week, and again the week after.

When those successive predictions disagree, the people and systems downstream have to react. Inventory gets re-ordered, shifts get re-staffed, capacity plans get rewritten. A model that keeps changing its mind imposes a real cost even when it is accurate on average.

Accuracy metrics cannot see this. `MAE`, `sCRPS` and friends score each forecast against the truth and average, which discards the relationship *between* forecasts of the same date. Two models can post identical accuracy while one is steady and the other thrashes.

This tutorial covers two metrics that measure that behaviour, and shows how to compute them from `cross_validation` output.

## The metrics

Both metrics compare pairs of forecasts that target the **same date** but were issued from different origins. Writing $\hat{y}^{before}$ for the forecast from the earlier origin and $\hat{y}^{update}$ for the one from the later origin:

**Forecast Percentage Change** measures how large the revisions are, and nothing else:

$$\mathrm{sFPC} = 200 \cdot \mathrm{mean}\left(\frac{|\hat{y}^{update} - \hat{y}^{before}|}{|\hat{y}^{update}| + |\hat{y}^{before}| + \epsilon}\right)$$

**Excess Volatility** asks whether a revision was *earned*. It charges the revision for its size and credits it for the accuracy it bought:

$$\mathrm{EV} = \underbrace{\mathrm{QL}(\hat{y}^{update}, \hat{y}^{before})}_{\text{cost of revising}} - \underbrace{\Big(\mathrm{QL}(y, \hat{y}^{before}) - \mathrm{QL}(y, \hat{y}^{update})\Big)}_{\text{accuracy gained}}$$

Because the pinball loss $\mathrm{QL}$ obeys the triangle inequality, the accuracy a revision buys can never exceed what the revision costs. So $\mathrm{EV} \geq 0$ always, and $\mathrm{EV} = 0$ exactly when a revision moves the forecast straight onto the truth. Anything above zero is churn the model did not pay for.

Both metrics need **overlapping** forecast windows. If the step between origins equals or exceeds the horizon, each date is predicted only once and there is nothing to compare.

### Setup

In [ ]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np

from neuralforecast import NeuralForecast
from neuralforecast.losses.numpy import (
    cross_validation_to_windows,
    excess_volatility,
    forecast_percentage_change,
)
from neuralforecast.losses.pytorch import MQLoss
from neuralforecast.models import MLP, NHITS
from neuralforecast.utils import AirPassengersPanel

warnings.filterwarnings("ignore")
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

## Two forecasters that accuracy cannot tell apart

To see why a stability metric is needed at all, we build two forecasters by hand and give them the *same* accuracy.

The series is a week of hourly demand with a daily shape and a weekday/weekend effect. We forecast it from six origins spaced 24 hours apart, each with a 48 hour horizon, so every window overlaps the next by 24 hours.

In [ ]:
HORIZON = 48
STEP_SIZE = 24
N_HOURS = 168
ORIGINS = [0, 24, 48, 72, 96, 120]

QUANTILES = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9]
BAND = 3.5
# half-width of each quantile relative to the median, as a multiple of BAND
SPREAD = {0.1: 1.0, 0.25: 0.55, 0.4: 0.25, 0.5: 0.0, 0.6: -0.25, 0.75: -0.55, 0.9: -1.0}

hours = np.arange(N_HOURS)


def daily_shape(h):
    t = (h % 24) / 24
    return (
        40
        + 18 * np.sin(np.pi * (t - 0.05))
        + 10 * np.exp(-((t - 0.83) ** 2) / 0.01)
        - 6 * np.exp(-((t - 0.15) ** 2) / 0.008)
    )


weekday_effect = np.array([1.0, 1.02, 1.03, 1.02, 1.04, 0.92, 0.88])[hours // 24]
truth = daily_shape(hours) * weekday_effect + np.random.default_rng(7).normal(0, 0.5, N_HOURS)

The two forecasters differ only in **where their error comes from**.

The *steady* forecaster is wrong in a way that persists: it draws one error signal for the whole week, so every origin is wrong about a given date in the same direction. Its forecasts for a shared date barely move.

The *erratic* forecaster draws fresh, independent error at every origin, and additionally flips the sign of a shift on the overlapping half of each window. Its forecasts for a shared date swing back and forth.

In [ ]:
def build_quantiles(medians):
    """Stack per-origin medians into the [B, T, H, C] and [B, T, H, C, Q] arrays the metrics take."""
    y = np.stack([truth[s : s + HORIZON] for s in ORIGINS])[None, :, :, None]
    y_hat = np.empty((1, len(ORIGINS), HORIZON, 1, len(QUANTILES)))
    for i, q in enumerate(QUANTILES):
        y_hat[0, :, :, 0, i] = np.stack(medians) - SPREAD[q] * BAND
    return y, y_hat


# steady: one persistent error signal shared by every origin, plus a little jitter
rng = np.random.default_rng(13)
persistent_error = rng.normal(0, 2.65, N_HOURS)
steady_medians = [
    truth[s : s + HORIZON] + persistent_error[s : s + HORIZON] + rng.normal(0, 0.8, HORIZON)
    for s in ORIGINS
]

# erratic: fresh error each origin, plus a sign-flipping shift on the overlapping half
rng = np.random.default_rng(77)
erratic_medians = []
for i, s in enumerate(ORIGINS):
    shift = np.zeros(HORIZON)
    shift[HORIZON // 2 :] = (-1) ** i * 3.0
    erratic_medians.append(truth[s : s + HORIZON] + rng.normal(0, 2.0, HORIZON) + shift)

y_steady, y_hat_steady = build_quantiles(steady_medians)
y_erratic, y_hat_erratic = build_quantiles(erratic_medians)

Plotting the six forecast windows against the truth makes the difference obvious.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, sharey=True)
colors = plt.cm.viridis(np.linspace(0, 0.9, len(ORIGINS)))

for ax, medians, title in [
    (axes[0], steady_medians, "Steady forecaster"),
    (axes[1], erratic_medians, "Erratic forecaster"),
]:
    ax.plot(hours, truth, color="black", linewidth=1.2, label="actual", zorder=5)
    for i, (start, median) in enumerate(zip(ORIGINS, medians)):
        window = np.arange(start, start + HORIZON)
        ax.plot(window, median, color=colors[i], linewidth=1.6, alpha=0.9)
        ax.fill_between(window, median - BAND, median + BAND, color=colors[i], alpha=0.12, linewidth=0)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_ylabel("demand")

axes[0].legend(loc="upper right", fontsize=9)
axes[1].set_xlabel("hour")
plt.tight_layout()
plt.show()

Now score them. First on accuracy:

In [ ]:
def accuracy(y, y_hat):
    median = y_hat[..., QUANTILES.index(0.5)]
    mae = np.abs(median - y).mean()
    errors = y[..., None] - y_hat
    q = np.array(QUANTILES)
    scrps = np.maximum(q * errors, (q - 1) * errors).mean()
    return mae, scrps


for name, y, y_hat in [("steady", y_steady, y_hat_steady), ("erratic", y_erratic, y_hat_erratic)]:
    mae, scrps = accuracy(y, y_hat)
    print(f"{name:8s}  MAE={mae:6.3f}  sCRPS={scrps:6.3f}")

The two are effectively tied: any accuracy-based comparison would call this a wash. Now the volatility metrics:

In [ ]:
for name, y, y_hat in [("steady", y_steady, y_hat_steady), ("erratic", y_erratic, y_hat_erratic)]:
    ev = excess_volatility(
        y=y, y_hat=y_hat, quantiles=QUANTILES, stride=STEP_SIZE, scaling=True
    )
    sfpc = forecast_percentage_change(
        y_hat=y_hat[..., QUANTILES.index(0.5)], stride=STEP_SIZE
    )
    print(f"{name:8s}  EV={ev:7.4f}  sFPC={sfpc:6.3f}")

The erratic forecaster revises its predictions several times as much as the steady one, and its excess volatility is roughly double: the revisions it makes are not buying it accuracy. Neither fact is visible in `MAE` or `sCRPS`.

This is the case for tracking stability alongside accuracy rather than instead of it. The two measure different things, and a model can be good at one and bad at the other.

## Using the metrics on `cross_validation` output

In practice the forecasts come from `NeuralForecast.cross_validation`, which returns a long DataFrame rather than the dense arrays above. `cross_validation_to_windows` bridges the two.

The one requirement is that **`step_size` must be smaller than the horizon**, so that windows overlap.

In [ ]:
df = AirPassengersPanel[["unique_id", "ds", "y"]]
HORIZON_AP = 12

models = [
    NHITS(h=HORIZON_AP, input_size=24, max_steps=100, loss=MQLoss(level=[80]),
          enable_progress_bar=False, logger=False, random_seed=0),
    MLP(h=HORIZON_AP, input_size=24, max_steps=100, loss=MQLoss(level=[80]),
        enable_progress_bar=False, logger=False, random_seed=0),
]

nf = NeuralForecast(models=models, freq="ME")
cv_df = nf.cross_validation(df=df, n_windows=8, step_size=4)
cv_df.head()

`cross_validation_to_windows` reads one model's forecast columns out of that frame and returns everything the metrics need: the dense arrays, the quantile levels recovered from the `-lo-80` / `-median` / `-hi-80` column names, a mask, and the step size inferred from the cutoffs.

In [ ]:
windows = cross_validation_to_windows(cv_df, model="NHITS")

print("quantiles:", windows.quantiles)
print("stride:   ", windows.stride)
print("y:        ", windows.y.shape, "  (series, windows, horizon, targets)")
print("y_hat:    ", windows.y_hat.shape)

With that in hand, scoring each model is a few lines:

In [ ]:
for model in ("NHITS", "MLP"):
    w = cross_validation_to_windows(cv_df, model=model)
    median = w.y_hat[..., w.quantiles.index(0.5)]

    mae = np.abs((median - w.y) * w.mask).sum() / w.mask.sum()
    ev = excess_volatility(
        y=w.y, y_hat=w.y_hat, quantiles=w.quantiles, stride=w.stride, mask=w.mask
    )
    sfpc = forecast_percentage_change(y_hat=median, stride=w.stride, mask=w.mask)

    print(f"{model:6s}  MAE={mae:7.3f}  EV={ev:7.4f}  sFPC={sfpc:6.3f}")

Read these together rather than separately. Accuracy tells you how close the forecasts land; `sFPC` tells you how much the model moves them between runs; `EV` tells you how much of that movement was wasted.

A few practical notes:

- `excess_volatility` needs a probabilistic forecast, so cross-validate with `level=` or `quantiles=`, or with a probabilistic loss such as `MQLoss`. `forecast_percentage_change` is a point metric and takes the median slice.
- `scaling=True` (the default) divides by the summed magnitude of the target, which makes `EV` comparable across series on different scales.
- Both metrics also accept plain arrays, so multivariate forecasts and forecasts produced outside `cross_validation` work too. The channel axis is always 1 on the `cross_validation` path, since it forecasts a single target column.

## References

- Willa Potosnak, Malcolm Wolff, Mengfei Cao, Ruijun Ma, Tatiana Konstantinova, Dmitry Efimov, Michael W. Mahoney, Boris Oreshkin, Kin G. Olivares. [Forking-Sequences: Statistically and Computationally Efficient Multi-Horizon Forecasting with Reduced Volatility](https://openreview.net/forum?id=dXdycy7WCX). Transactions on Machine Learning Research (2026).